# LLM for Two Stage problem 

1. two stage sp
2. two stage ro
3. two stage dro 

# Take two stage sp (MILP) for example 

$$
\begin{align}
\min \ & c^T x + \mathbb{E}\left[ Q(x, \xi) \right] \\
\text{s.t.} & A x = b, \\
& x \geq 0. 
\end{align}
$$

$$
\begin{align}
Q(x, \xi) = \min \ & q(\xi)^T y \\
\text{s.t.} & T(\xi) x + W(\xi) y = h(\xi), \\
& y \geq 0.  
\end{align}
$$

In [16]:
# export HF_ENDPOINT=https://hf-mirror.com  # 设置hf的镜像 

# nohup 放在后台运行，vllm server 部署 “”的模型，开启自动工具选择，工具调用解析器使用hermes，推理解析器使用deepseek_r1，日志输出到vllm_qwen3_8b.log，错误输出也重定向到日志文件中。
# nohup vllm serve "Qwen/Qwen3-8B" --enable-auto-tool-choice --tool-call-parser hermes --reasoning-parser deepseek_r1  > vllm_qwen3_8b.log 2>&1 &
 
# 获取后台运行的进程ID，并将其写入vllm_qwen3_8b.pid文件中。
# echo $! > vllm_qwen3_8b.pid 

# 使用存储在vllm_qwen3_8b.pid文件中的进程ID终止该进程。
# kill `cat vllm_qwen3_8b.pid`


# curl 测试部署的模型接口api 
# curl -X POST "http://localhost:8000/v1/chat/completions" \
#   -H "Content-Type: application/json" \
#   --data '{
#       "model": "Qwen/Qwen3-8B",
#       "messages": [
#           {
#               "role": "user",
#               "content": "What is the capital of France?"
#           }
#       ]
#   }'

# curl -X POST "https://1yvxf19722895.vicp.fun/v1/chat/completions" \
#   -H "Content-Type: application/json" \
#   --data '{
#       "model": "Qwen/Qwen3-8B",
#       "messages": [
#           {
#               "role": "user",
#               "content": "What is the capital of France?"
#           }
#       ]
#   }'



import numpy as np 
import random 

np.random.seed(0)

range_of_coef_min = -100
range_of_coef_max = 100 
range_of_y_min = -100
range_of_y_max = 100

x_dim = 10 
num_x_cons = 5 

c = np.random.randint(range_of_coef_min, range_of_coef_max, size=(x_dim,)) 
A = np.random.randint(range_of_coef_min, range_of_coef_max, size=(num_x_cons, x_dim)) 
b = np.random.randint(range_of_coef_min, range_of_coef_max, size=(num_x_cons,)) 
# print(c.shape)
# print(A.shape)
# print(b.shape)
num_snr = 4 
y_dim = 20 
num_y_cons = 10
q = np.random.randint(range_of_coef_min, range_of_coef_max, size=(num_snr, y_dim,))
T = np.random.randint(range_of_coef_min, range_of_coef_max, size=(num_snr, num_y_cons, x_dim)) 
W = np.random.randint(range_of_coef_min, range_of_coef_max, size=(num_snr, num_y_cons, y_dim))
h = np.random.randint(range_of_coef_min, range_of_coef_max, size=(num_snr, num_y_cons,))
# print(q.shape)
# print(T.shape)
# print(W.shape)
# print(h.shape)
print("c:", c)
print("A:", A)
print("b:", b)
print("q:", q)
print("T:", T)
print("W:", W)
print("h:", h)

c: [ 72 -53  17  92 -33  95   3 -91 -79 -64]
A: [[-13 -30 -12  40 -42  93 -61 -13  74 -12]
 [-19  65 -75 -23 -28 -91  48  15  97 -21]
 [ 75  92 -18  -1  77 -71  47  47  42  67]
 [-68  93 -91  85  27 -68 -69  51  63  14]
 [ 83 -72 -66  28  28  64 -47  33 -62 -83]]
b: [-21  32   5 -58  86]
q: [[ -69   20  -99  -35   69  -43  -65    2   19  -89   74  -18   -9   28
    42   -1  -47   40   21   70]
 [ -16  -32  -94   96  -53   27   31    0   80  -22   43   48   86  -77
    41   17  -15  -52  -51  -31]
 [  69   63   92   -5   97   -6 -100   13   78  -64   62  -52   -7   31
    -2  -58   12   49   27 -100]
 [  38   14  -57   86   27  -77   87   30   21   -2  -38   63   23   95
   -18   74   48  -50   55  -86]]
T: [[[ -59  -42   93  -64  -90  -14  -57    4  -89  -98]
  [ -49  -20  -68   82   28  -62  -81   74  -58   15]
  [  84   88  -23  -70  -76   25  -98  -97   -6    7]
  [ -87   12  -60  -28  -81   -5  -28   54   94   80]
  [ -33  -39  -86   -4  -96   95   39  -14   21    9]
  [ -25   84  

In [17]:
from gurobipy import Model, GRB

m = Model("two_stage_SP_DE")

x = m.addMVar(x_dim, vtype=GRB.BINARY, name="x")

y = []
for s in range(num_snr):
    y.append(m.addMVar(y_dim, lb=range_of_y_min, ub=range_of_y_max, vtype=GRB.CONTINUOUS, name=f"y_{s}"))

# 目标函数
obj = c @ x
for s in range(num_snr):
    obj += (1/num_snr) * (q[s] @ y[s])
m.setObjective(obj, GRB.MINIMIZE)

# 第一阶段约束 A x == b
# m.addConstr(A @ x == b, name="Ax_eq_b")

# 第二阶段约束 T x + W y == h
for s in range(num_snr):
    m.addConstr(T[s] @ x + W[s] @ y[s] == h[s], name=f"snr_{s}_TxWy_eq_h")


m.optimize()
if m.status == GRB.OPTIMAL:
    print("Objective:", m.objVal)
    print("Optimal x:", x.X)
    print("Optimal y for each scenario:")
    for s in range(num_snr):
        print(f"y_{s}:", y[s].X)
elif m.status == GRB.INFEASIBLE:
    print("模型不可行")
else:
    print("优化未得到最优解，状态码:", m.status)

Set parameter Username
Academic license - for non-commercial use only - expires 2027-01-21
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 25.2.0 25C56)

CPU model: Apple M1 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 40 rows, 90 columns and 1190 nonzeros
Model fingerprint: 0x7c9d1412
Variable types: 80 continuous, 10 integer (10 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+02]
  Objective range  [2e-01, 1e+02]
  Bounds range     [1e+00, 1e+02]
  RHS range        [1e+00, 1e+02]
Presolve time: 0.00s
Presolved: 40 rows, 90 columns, 1143 nonzeros
Variable types: 80 continuous, 10 integer (10 binary)
Found heuristic solution: objective -60602.01244

Root relaxation: objective -6.061488e+04, 50 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    

In [24]:

# def solve_second_stage(x_val: np.ndarray) -> dict:
#     total_cost = 0
#     snr_cost = []
#     optimal_y = []
#     for s in range(num_snr):
#         sub_m = Model(f"second_stage_snr_{s}")
#         sub_m.setParam("OutputFlag", 0)
#         y_s = sub_m.addMVar(y_dim, lb=range_of_y_min, ub=range_of_y_max, vtype=GRB.CONTINUOUS, name=f"y_{s}")
#         sub_m.setObjective(q[s] @ y_s, GRB.MINIMIZE)
#         sub_m.addConstr(T[s] @ x_val + W[s] @ y_s == h[s], name=f"snr_{s}_TxWy_eq_h")
#         sub_m.optimize()
#         if sub_m.status == GRB.OPTIMAL:
#             total_cost += sub_m.objVal
#             snr_cost.append(sub_m.objVal)
#             optimal_y.append(y_s.X)
#         else:
#             print(f"Scenario {s} second stage problem not optimal, status code:", sub_m.status)
#     return {
#         "second_stage_cost": total_cost / num_snr,
#         "snr_costs": snr_cost,
#         "optimal_y": optimal_y
#     }
import json 

def solve_second_stage(x_val: list[int]) -> str:
    """求解两阶段优化问题的第二阶段问题，输入为第一阶段整数解x_val，输出为第二阶段问题的最优值和最优解的json字符串。"""
    np.array(x_val)
    total_cost = 0
    snr_cost = []
    optimal_y = []
    for s in range(num_snr):
        sub_m = Model(f"second_stage_snr_{s}")
        sub_m.setParam("OutputFlag", 0)
        y_s = sub_m.addMVar(y_dim, lb=range_of_y_min, ub=range_of_y_max, vtype=GRB.CONTINUOUS, name=f"y_{s}")
        sub_m.setObjective(q[s] @ y_s, GRB.MINIMIZE)
        sub_m.addConstr(T[s] @ x_val + W[s] @ y_s == h[s], name=f"snr_{s}_TxWy_eq_h")
        sub_m.optimize()
        if sub_m.status == GRB.OPTIMAL:
            total_cost += sub_m.objVal
            snr_cost.append(sub_m.objVal)
            optimal_y.append(y_s.X.tolist())
        else:
            print(f"Scenario {s} second stage problem not optimal, status code:", sub_m.status)
    rt_dict = {
        "second_stage_cost": total_cost / num_snr,
        "snr_costs": snr_cost,
        "optimal_y": optimal_y
    }
    return json.dumps(rt_dict)

In [25]:
x_val = [-0., 1., 1., -0., 1., -0., -0., 1., 1., 1.] # np.array([-0., 1., 1., -0., 1., -0., -0., 1., 1., 1.])
rt = solve_second_stage(x_val)
print(rt)

{"second_stage_cost": -60311.88090936022, "snr_costs": [-66139.497138604, -54430.54059458196, -57347.82915049433, -63329.65675376057], "optimal_y": [[100.0, -100.0, 87.12947569118408, 49.40004061959045, -100.0, 100.0, 100.0, -62.129368668060934, -100.0, 100.0, -74.96487852542154, 100.0, 53.674704345350115, 27.112391216593767, 67.2593338132625, -52.429696550749675, 100.0, -92.23527489231157, 62.99984708354343, -100.0], [100.0, -75.51552037624218, 87.37179338639861, -100.0, 100.0, -100.0, -73.35027012425364, -100.0, -100.0, 24.953814818053978, 88.2934321269393, -100.0, -100.0, 71.0756712946029, -81.87535701266096, 23.226086254368003, -49.72444150417172, 100.0, -15.316515458962796, -100.0], [-32.49851447727186, -100.0, -100.0, 99.09868383840606, -64.9088846682671, -100.0, 100.0, 4.143003870519555, -100.0, 14.288565642984075, 8.036181654506969, 100.0, -100.0, 100.0, -65.03942226846493, 100.0, 100.0, -1.6517196557208076, 17.346086428936918, 97.68899446281748], [17.971187040154902, 0.0241985

In [2]:

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI


model = ChatOpenAI(
    base_url="https://1yvxf19722895.vicp.fun/v1",  
    api_key="xxx",                 
    model="Qwen/Qwen3-8B",
)

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)


{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='7449c8ef-1e06-4a84-9b4a-2aff046670f6'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 329, 'prompt_tokens': 158, 'total_tokens': 487, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-8B', 'system_fingerprint': None, 'id': 'chatcmpl-96b3863276011b09', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c2b82-d245-7530-a728-f3a8f7afb892-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'chatcmpl-tool-8a636c3615151c04', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 158, 'output_tokens': 329, 'total_tokens': 487, 'input_token_details': {}, 'output_token_details': {}}),
  ToolMessage(content="It's always sunny in San Francisco!", name='get_weather', id='11d487bc-da9

In [8]:
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

In [9]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

In [10]:
from langchain_openai import ChatOpenAI


model = ChatOpenAI(
    base_url="https://1yvxf19722895.vicp.fun/v1",  
    api_key="xxx",                 
    model="Qwen/Qwen3-8B",
)

In [11]:
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    # A punny response (always required)
    punny_response: str
    # Any interesting information about the weather if available
    weather_conditions: str | None = None

In [12]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location], 
    context_schema=Context, # 提供不变的上下文信息 如userid
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer # 默认用于 存储 历史messages
)

# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     weather_conditions="It's always sunny in Florida!"
# )


# Note that we can continue the conversation using the same `thread_id`.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="You're 'thund-erfully' welcome! It's always a 'breeze' to help you stay 'current' with the weather. I'm just 'cloud'-ing around waiting to 'shower' you with more forecasts whenever you need them. Have a 'sun-sational' day in the Florida sunshine!",
#     weather_conditions=None
# )

ResponseFormat(punny_response="Why don't Floridians ever get cold? They've got a 'sun' of it!", weather_conditions="It's always sunny in Florida!")
ResponseFormat(punny_response="You're welcome! Hope your day is as bright as Florida's sunshine!", weather_conditions=None)
